# Delta FullCopy Ingestion Pipeline

**Notebook Type:** Bronze Layer Ingestion  
**Data Source:** Excel Files (Marcobre Fileserver)  
**Target:** Delta Lake (Unity Catalog)  
**Update Mode:** Full Overwrite (Full Copy Pattern)

## Overview
This notebook performs full copy ingestion of Excel files into Delta Lake tables. It supports both vertical and horizontal data orientations, multi-level headers, custom column mapping, and metadata extraction (report dates and forecast names). The notebook reads from the Bronze pre-landing zone and writes to the Bronze landing zone with schema evolution enabled.

## Prerequisites
- **Functions Notebook**: Requires `./Functions` notebook with utility functions (`parse_excel_range_advanced`, `apply_datetime_tokens`, `build_headers`, `sanitize_column`, `fill_blank_columns`, `fix_duplicate_columns`, `normalize_dataframe_types`, `drop_all_nan_rows`, `slugify`)
- **Widgets**: All parameters must be provided via notebook widgets (ExcelPath, TableName, SheetName, DataRange, etc.)
- **Volumes**: Access to `{environment}_bronze_prelanding_minsur` and `{environment}_bronze_landing_minsur` Unity Catalog volumes
- **Libraries**: `pandas`, `openpyxl`, `delta-spark`

## 1. Import Shared Functions
Load common utility functions from the Functions notebook (parse_excel_range_advanced, apply_datetime_tokens, sanitize_column, etc.)

In [0]:
%run ./Functions

## 2. Library Imports and Spark Configuration
Import required Python libraries and configure Spark settings for Excel processing

In [0]:
import pandas as pd
import numpy as np
import re
import unicodedata
import warnings
from pathlib import Path
from collections import defaultdict
from datetime import datetime
from delta.tables import DeltaTable
from pyspark.sql.utils import AnalysisException

# Disable Arrow optimization to avoid Spark-Pandas compatibility issues during DataFrame conversion
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "false")

# Suppress openpyxl warnings about Excel file formatting that don't affect data reading
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    module="openpyxl"
)

## 3. Define Notebook Widgets (Parameters)
Create interactive widgets to accept configuration parameters for the ingestion process.
These values will be passed from the orchestration pipeline.

In [0]:
# Environment identifier (dev, test, prod) used for volume path construction
dbutils.widgets.text("Environment", "", "Environment")

# Source Excel file path (supports datetime tokens like {yyyyMMddHH})
dbutils.widgets.text("ExcelPath", "", "Excel Path")

# Target Delta table name in the catalog
dbutils.widgets.text("TableName", "", "Table Name")

# Specific sheet name within the Excel workbook
dbutils.widgets.text("SheetName", "", "Sheet Name")

# Cell range defining the data boundaries (e.g., "A1:Z100")
dbutils.widgets.text("DataRange", "", "Data Range")

# Column indices to extract when using custom column mapping (comma-separated)
dbutils.widgets.text("IndexColumns", "", "Index Columns")

# Custom column names to assign when using index-based extraction (comma-separated)
dbutils.widgets.text("CustomColumnNames", "", "Custom Column Names")

# Cell reference containing the report date value (e.g., "B2")
dbutils.widgets.text("ReportDate", "", "Report Date")

# Cell reference containing the forecast identifier (e.g., "C1")
dbutils.widgets.text("ForecastName", "", "Forecast Name")

# Column ranges to skip during reading (e.g., "B:D,F")
dbutils.widgets.text("ColumnsToSkip", "", "Columns To Skip")

# Row ranges to skip during reading (e.g., "1:3,5")
dbutils.widgets.text("RowsToSkip", "", "Rows To Skip")

# Flag to indicate if data is horizontally oriented (requires transpose)
dbutils.widgets.text("IsHorizontal", "", "Is Horizontal")

# Number of header rows to merge for multi-level Excel headers
dbutils.widgets.text("MergeNHeaders", "", "Merge N Headers")

# Execution timestamp for file path token replacement (defaults to current UTC time)
dbutils.widgets.text("Datetime", "", "Datetime")

## 4. Retrieve Widget Values and Construct Paths
Extract parameter values from widgets and build the final file path using datetime token replacement

In [0]:
# Retrieve configuration values from notebook widgets
excel_file_path = dbutils.widgets.get("ExcelPath")
table_name = dbutils.widgets.get("TableName")
excel_sheet_name = dbutils.widgets.get("SheetName")
excel_cell_range = dbutils.widgets.get("DataRange")
index_columns = dbutils.widgets.get("IndexColumns")
custom_column_names = dbutils.widgets.get("CustomColumnNames")
report_date = dbutils.widgets.get("ReportDate")
forecast_name = dbutils.widgets.get("ForecastName")
skip_col_range = dbutils.widgets.get("ColumnsToSkip")
skip_row_range = dbutils.widgets.get("RowsToSkip")

# Convert string boolean to actual boolean for orientation flag
is_horizontal = (True if str(dbutils.widgets.get("IsHorizontal")).lower() == "true" else False)

# Parse number of header rows to merge (default to 1 if not specified)
header_rows = int(dbutils.widgets.get("MergeNHeaders") or 1)

# Get execution datetime or default to current UTC time
current_datetime = str(dbutils.widgets.get("Datetime") or datetime.now("UTC").strftime("%Y-%m-%d %H:%M:%S"))

# Get environment identifier for volume path construction
environment = dbutils.widgets.get("Environment")

# Apply datetime token replacement to file path (e.g., {yyyyMMddHH} -> 2024012414)
excel_file_path = excel_file_path.replace(".xlsx", "_{yyyyMMddHH}.xlsx")
final_path = apply_datetime_tokens(excel_file_path, datetime.strptime(current_datetime, "%Y-%m-%d %H:%M:%S"))

# Construct full DBFS volume path using Unity Catalog external volume
input_excel_path = f"/Volumes/{environment}_bronze_prelanding_minsur/marcobre_fileserver/marcobre_fileserver/{final_path}"

# Debug output to verify path construction and orientation settings
print(f"Input path: {input_excel_path}")
print(f"Is horizontal: {is_horizontal}")
print(f"Header rows to merge: {header_rows}")

## 5. Parse Excel Range Configuration
Convert human-readable range specifications into pandas-compatible parameters

In [0]:
# Parse the Excel range string and skip configurations into structured parameters
# Returns dictionary with: usecols, skiprows, nrows for pandas.read_excel()
params = parse_excel_range_advanced(
    table_range=excel_cell_range,
    skip_col_range=skip_col_range,
    skip_row_range=skip_row_range
)

# Debug output to verify range parsing logic
print("Parsed Excel parameters:")
print(params)

## 6. Read Excel Data
Load data from Excel file with support for both horizontal (transposed) and vertical orientations

In [0]:
# Read raw data from Excel without header processing (header=None)
# We'll handle header construction manually to support multi-row headers
df = pd.read_excel(
    input_excel_path,
    sheet_name=excel_sheet_name,
    usecols=params['usecols'],
    skiprows=params['skiprows'],
    nrows=params['nrows'],
    header=None,  # Disable automatic header detection
    engine="openpyxl"
)

# Check if data is horizontally oriented (requires 90-degree rotation)
if is_horizontal:
    # Transpose the DataFrame to convert horizontal layout to vertical
    df_t = df.T.reset_index(drop=True)
    
    # Check if custom column mapping is specified (IndexColumns + CustomColumnNames)
    if index_columns != "" and custom_column_names != "":
        # Parse column indices and custom names from comma-separated strings
        col_indexes = [int(i) for i in index_columns.split(",")]
        new_column_names = [c.strip() for c in custom_column_names.split(",")]
        
        # Validate that indices count matches names count
        if len(col_indexes) != len(new_column_names):
            raise ValueError(
                f"Number of index columns ({len(col_indexes)}) does not match "
                f"number of custom column names ({len(new_column_names)})"
            )
        
        # Extract only specified columns and assign custom names
        df_final = df_t.iloc[:, col_indexes]
        df_final.columns = new_column_names
    else:
        # Build composite headers from first N rows using the build_headers utility function
        headers = build_headers(df_t, header_rows)
        df_t.columns = headers
        
        # Remove the header rows from the actual data
        df_final = df_t.iloc[header_rows:].reset_index(drop=True)
else:
    # Vertical orientation (standard Excel layout)
    if index_columns != "" and custom_column_names != "":
        # Parse column indices (1-based to 0-based conversion)
        col_indexes = [int(i) - 1 for i in index_columns.split(",")]
        new_column_names = [c.strip() for c in custom_column_names.split(",")]
        
        # Validate mapping configuration
        if len(col_indexes) != len(new_column_names):
            raise ValueError(
                f"Number of index columns ({len(col_indexes)}) does not match "
                f"number of custom column names ({len(new_column_names)})"
            )
        
        # Extract specified columns with custom naming
        df_final = df.iloc[:, col_indexes]
        df_final.columns = new_column_names
    else:
        # Build headers from first N rows for multi-level header support
        headers = build_headers(df, header_rows)
        df.columns = headers
        
        # Remove header rows from data and reset index
        df = df.iloc[header_rows:].reset_index(drop=True)
        df_final = df.copy()

## 7. Extract Metadata Fields
Optionally extract report date and forecast name from specific cells in the Excel file

In [0]:
# Initialize metadata value containers
report_date_val = ""
forecast_name_val = ""

# Extract report date if cell reference is provided
if report_date.strip():
    # Parse the cell reference (e.g., "B2" -> usecols and skiprows)
    report_params = parse_excel_range_advanced(f"{report_date.strip()}:{report_date.strip()}")
    
    # Read the single cell containing the report date
    report_df = pd.read_excel(
        input_excel_path,
        sheet_name=excel_sheet_name,
        usecols=report_params['usecols'],
        skiprows=report_params['skiprows'],
        nrows=report_params['nrows'],
        header=None,
        engine="openpyxl"
    )
    
    # Extract the scalar value from the single-cell DataFrame
    report_date_val = str(report_df.iloc[0, 0])
    
    # Add report_date column to the final dataset
    df_final = df_final.copy()
    df_final.loc[:, "report_date"] = report_date_val

# Extract forecast name if cell reference is provided
if forecast_name.strip():
    # Parse the cell reference for forecast name
    forecast_params = parse_excel_range_advanced(f"{forecast_name.strip()}:{forecast_name.strip()}")
    
    # Read the single cell containing the forecast identifier
    forecast_df = pd.read_excel(
        input_excel_path,
        sheet_name=excel_sheet_name,
        usecols=forecast_params['usecols'],
        skiprows=forecast_params['skiprows'],
        nrows=forecast_params['nrows'],
        header=None,
        engine="openpyxl"
    )
    
    # Extract forecast name value
    forecast_name_val = str(forecast_df.iloc[0, 0])
    
    # Add forecast column to the final dataset
    df_final = df_final.copy()
    df_final.loc[:, "forecast"] = forecast_name_val

## 8. Data Cleaning and Transformation
Sanitize column names, handle duplicates, normalize data types, and remove empty rows

In [0]:
# Store original column names for transformation tracking and debugging
original_columns = df_final.columns.tolist()

# Sanitize column names: lowercase, replace special chars with underscores, normalize accents
df_final.columns = [sanitize_column(c) for c in df_final.columns]

# Fill blank/empty column names with generated names (unnamed_1, unnamed_2, etc.)
df_final = fill_blank_columns(df_final)

# Resolve duplicate column names by appending numeric suffixes (col, col_1, col_2, etc.)
df_final.columns = fix_duplicate_columns(df_final.columns)

# Normalize data types: convert strings to numeric where possible, standardize dates, etc.
df_final = normalize_dataframe_types(df_final)

# Remove rows where all values are NaN (completely empty rows)
df_final = drop_all_nan_rows(df_final)

# Prepare forecast name suffix for table naming (if applicable)
final_forecast_name = ""
if forecast_name_val:
    final_forecast_name = f"_{forecast_name_val.lower().replace(' ', '_')}"

# Display column name transformation mapping for audit purposes
print("Column mapping (Original -> Sanitized):")
for original, sanitized in zip(original_columns, df_final.columns):
    print(f"  {original}  -->  {sanitized}")

## 9. Convert to Spark DataFrame and Preview
Convert the cleaned pandas DataFrame to Spark DataFrame for Delta Lake operations

In [0]:
# Convert pandas DataFrame to Spark DataFrame for distributed processing
df_table = spark.createDataFrame(df_final)

# Display preview of the data (first 1000 rows) for validation
display(df_table)

## 10. Write to Delta Lake
Save the processed data as a Delta table in the Bronze layer with schema evolution enabled

In [0]:
# Normalize sheet name for table naming (remove spaces, dots, slashes)
final_excel_sheet = excel_sheet_name.lower().replace(' ', '_').replace('.', '').replace('/', '_')

# Normalize table name using slugify utility to ensure valid identifier
final_destination_table = f"{slugify(table_name.lower().replace(' ', '_').replace('.', '').replace('/', '_'))}"
new_table_name = f"{final_destination_table}"

# Write Spark DataFrame to Delta table in Unity Catalog
# - format("delta"): Use Delta Lake format for ACID transactions
# - option("mergeSchema", "true"): Enable schema evolution to add new columns automatically
# - mode("overwrite"): Replace existing table contents (full copy pattern)
df_table.write \
  .format("delta") \
  .option("mergeSchema", "true") \
  .mode("overwrite") \
  .saveAsTable(f"{environment}_bronze_landing_minsur.marcobre_fileserver.{new_table_name}")

# Log the destination for pipeline monitoring and debugging
print(f"Delta table successfully written to: {environment}_bronze_landing_minsur.marcobre_fileserver.{new_table_name}")